In [2]:
!pip install chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [3]:
!pip install sentence-transformers chromadb groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
import chromadb
from groq import Groq
import os
print("All Packages imported successfully")

All Packages imported successfully


In [5]:
GROQ_API_KEY = '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'
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized.")
print("Note: If you see an authentication error later, double click your API key.")

Groq API client initialized.
Note: If you see an authentication error later, double click your API key.


In [6]:
df = pd.read_csv("/content/drive/MyDrive/15 days intern dataset/college_notes.csv")
print("=== Data Loaded Successfully ===")
print()
print("Shape the DataSet : ",df.shape)
print()
print("Column Names : ",df.columns.to_list())
print()
print("First 3 Rows : ",df.head(3))

=== Data Loaded Successfully ===

Shape the DataSet :  (15, 4)

Column Names :  ['note_id', 'subject', 'topic', 'content']

First 3 Rows :    note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [7]:
print("Subject in the datasets : ")
print(df['subject'].value_counts())
print("-"*40)
print("Sample of topics : ")
print(df[['note_id','subject','topic']].to_string(index=False))
print("-"*40)
print("Length of content (number of characters) : for each note : ")
df['content_length'] = df['content'].apply(lambda x: len(x))
print(df[['note_id','content_length']].to_string(index=False))

Subject in the datasets : 
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64
----------------------------------------
Sample of topics : 
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI R

In [8]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {
        "Subjects": row['subject'] , "topic": row['topic']
    }
    for row in df.to_dict('records')
]
print("Total chunks prepared:", len(documents))
print("first document id:", ids[0])
print("First metadata:", metadatas[0])
print("First 100 chars od doc:", documents[0][:100])

Total chunks prepared: 15
first document id: note_N001
First metadata: {'Subjects': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars od doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc


In [9]:
from sentence_transformers import SentenceTransformer

print("Loading Embedding model....")
print("This may 30-60 seconds on first run - model is being doenloaded")
print("Subsequent runs will be faster as the model is cached")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded successfully")
test_embedding = embedding_model.encode("This is a sentence")
print("Test embedding shape:", test_embedding.shape)
print("First 5 values of test embedding:", test_embedding[:5])

Loading Embedding model....
This may 30-60 seconds on first run - model is being doenloaded
Subsequent runs will be faster as the model is cached


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully
Test embedding shape: (384,)
First 5 values of test embedding: [ 0.05048309  0.088006    0.00487488  0.03626884 -0.00101813]


In [10]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("chromadb client created")
print("Collection name: college_notes_rag")
print("Documents in collection so far:", collection.count())

chromadb client created
Collection name: college_notes_rag
Documents in collection so far: 0


In [11]:
print("Generating embeddings for all 15 notes")
print("this may take 15-30 seconds .....")
embeddings = embedding_model.encode(documents,show_progress_bar = True)
print("Embedding matrix shape:", embeddings.shape)
embeddings_list = embeddings.tolist()
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas,

)
print("Documents successfully added to chromadb")
print("Total documents in collection:", collection.count())

Generating embeddings for all 15 notes
this may take 15-30 seconds .....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (15, 384)
Documents successfully added to chromadb
Total documents in collection: 15


In [12]:
def retrieve_relevant_chunks(question, top_k=3):

    # Convert question into embedding
    question_embedding = embedding_model.encode(
        question
    ).tolist()

    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    return results


print("Retrieval function created successfully")

print(
    "Function: retrieve_relevant_chunks(question, top_k=3)"
)

Retrieval function created successfully
Function: retrieve_relevant_chunks(question, top_k=3)


In [13]:
test_question = "What is ETL and how does it work in data engineering?"
print("Test question:", test_question)

results = retrieve_relevant_chunks(test_question,top_k=3)
print("Top 3 retrieved chunks:")

for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
   print("Result",i+1)
   print("Subject:", meta['Subjects'])
   print("Topic:", meta['topic'])
   print(f"Distance: {dist:.4f}")
   print("Content:", doc[:120])

Test question: What is ETL and how does it work in data engineering?
Top 3 retrieved chunks:
Result 1
Subject: Data Engineering
Topic: ETL Pipelines
Distance: 0.2269
Content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i
Result 2
Subject: Data Engineering
Topic: APIs and Data Collection
Distance: 1.0690
Content: An API or Application Programming Interface allows two software applications to talk to each other. In data engineering 
Result 3
Subject: Python Programming
Topic: Data Visualization
Distance: 1.3375
Content: Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo


In [1]:
def build_context_from_results(results):
    """
    Build formatted context from retrieved ChromaDB results
    """

    context_parts = []

    for i, (doc, meta) in enumerate(
        zip(
            results['documents'][0],
            results['metadatas'][0]
        )
    ):

        context_text = f"""
Result {i+1}

Subject: {meta['subject']}

Topic: {meta['topic']}

Content:
{doc}
"""

        context_parts.append(context_text)

    final_context = "\n" + ("-" * 50).join(context_parts)

    return final_context


print("Context Builder Function Created Successfully")

Context Builder Function Created Successfully


In [18]:
def generate_rag_answer(question,context):
  """
  send the retrieved context and question to groq llm for answer generation

  parameters:
  question(str): the user question
  context(str): the retrieved chunks

  Returns:
      answer(str): The LLM's generated answer
  """

  system_prompt = """
You are a helpful academic assistant for engineering students.

Your role is to answer student questions using only the provided context.

Guidelines:
1. Use the retrieved context to generate accurate answers.
2. If the answer is not available in the context, say:
   "I could not find relevant information in the notes."
3. Keep answers clear, concise, and beginner-friendly.
4. Explain technical concepts in simple language.
5. Use proper academic tone.
6. Do not generate unrelated or hallucinated information.
7. If multiple topics are retrieved, combine them logically.
8. Focus on engineering subjects such as:
   - Data Engineering
   - Machine Learning
   - Cloud Computing
   - Generative AI

You will receive:
- Retrieved context from semantic search
- A user question

Generate the best possible answer based on the context.
"""

  user_prompt = f"""
Context:
{context}

Question:
{question}

Answer the question using only the provided context.
Explain the answer clearly for an engineering student.
"""

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )

  answer = response.choices[0].message.content
  return answer

print("RAG generation function defined")

RAG generation function defined


In [20]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Complete RAG pipeline:
    Given a question, retrieve relevant context
    and generate an answer using the LLM.

    Parameters:
    question : str
        User question

    top_k : int
        Number of retrieved chunks

    verbose : bool
        Print intermediate outputs

    Returns:
    answer : str
        Final generated answer
    """

    # ============================================
    # STEP 1: PRINT QUESTION
    # ============================================

    if verbose:

        print("Question:", question)

        print("-" * 50)

        print(
            "Step 1 - Retrieving Relevant Documents...."
        )

    # ============================================
    # STEP 2: RETRIEVE RELEVANT CHUNKS
    # ============================================

    results = retrieve_relevant_chunks(
        question,
        top_k=top_k
    )

    # ============================================
    # STEP 3: PRINT RETRIEVED RESULTS
    # ============================================

    if verbose:

        print("Retrieved Documents:",
              len(results['documents'][0]))

        print("-" * 50)

    # ============================================
    # STEP 4: BUILD CONTEXT
    # ============================================

    context = build_context_from_results(
        results
    )

    if verbose:

        print("Step 2 - Building Context....")

        print("-" * 50)

        print(context)

        print("-" * 50)

    # ============================================
    # STEP 5: SYSTEM PROMPT
    # ============================================

    system_prompt = """
You are a helpful academic assistant for engineering students.

Use only the provided context to answer questions.

Rules:
1. Answer clearly and accurately.
2. Keep explanations simple and beginner-friendly.
3. If answer is unavailable in context,
   say:
   'I could not find relevant information in the notes.'
4. Do not generate fake or unrelated information.
"""

    # ============================================
    # STEP 6: USER PROMPT
    # ============================================

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer the question using only the provided context.
"""

    if verbose:

        print("Step 3 - Generating Final Answer....")

        print("-" * 50)

    # ============================================
    # STEP 7: GENERATE RESPONSE USING GROQ
    # ============================================

    response = client.chat.completions.create(

        model="llama3-8b-8192",

        messages=[

            {
                "role": "system",
                "content": system_prompt
            },

            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    # ============================================
    # STEP 8: EXTRACT FINAL ANSWER
    # ============================================

    answer = response.choices[0].message.content

    # ============================================
    # STEP 9: PRINT FINAL ANSWER
    # ============================================

    if verbose:

        print("Final Answer:")

        print("-" * 50)

        print(answer)

    # ============================================
    # STEP 10: RETURN ANSWER
    # ============================================

    return answer


print("Complete RAG Assistant Created Successfully")

Complete RAG Assistant Created Successfully
